Data leakage:


validation data accidentally affects training


causes unrealistically high metrics


Correct workflow:


split first


fit preprocessing on train only


apply transformations to val/test


Pipeline:


automates preprocessing + modeling safely


prevents leakage automatically


Main concept:
Validation data must remain completely unseen during training and preprocessing

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the RAW titanic data — not the cleaned version
df = pd.read_csv('train.csv')

# Minimal feature engineering only (no imputation yet — pipeline will handle it)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
df['Has_Cabin']  = df['Cabin'].notna().astype(int)
df['Title']      = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title']      = df['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare'
)
df['Title'] = df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
df['Sex']      = df['Sex'].map({'male':0,'female':1})
df['Embarked'] = df['Embarked'].map({'S':0,'C':1,'Q':2})
df['Title']    = df['Title'].map({'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Rare':4})

features = ['Pclass','Sex','Age','Fare','Embarked',
            'FamilySize','IsAlone','Has_Cabin','Title']
X = df[features]
y = df['Survived']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

This code:

starts from raw Titanic data---
performs safe feature engineering---
splits before preprocessing---
intentionally keeps missing values for pipeline handling---

Main concept:

Proper ML workflow separates train/validation before any learned preprocessing to avoid leakage

In [2]:
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_val)
print("Pipeline accuracy:", round(accuracy_score(y_val, y_pred), 4))
print(classification_report(y_val, y_pred))

Pipeline accuracy: 0.8101
              precision    recall  f1-score   support

           0       0.84      0.85      0.85       110
           1       0.76      0.74      0.75        69

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [3]:
print(pipeline.named_steps)
print(pipeline.named_steps['imputer'].statistics_)  # medians learned from train
print(pipeline.named_steps['model'].feature_importances_)

{'imputer': SimpleImputer(strategy='median'), 'scaler': StandardScaler(), 'model': RandomForestClassifier(random_state=42)}
[ 3.      0.     28.5    14.4542  0.      1.      1.      0.      0.    ]
[0.07233954 0.16646782 0.21425851 0.22714231 0.02938648 0.06497651
 0.01248042 0.05207486 0.16087355]


Pipeline:

combines preprocessing + model into one workflow

prevents leakage automatically

simplifies ML code

Pipeline steps:

impute missing values

scale features

train model

Useful methods:

named_steps → inspect components

statistics_ → learned medians

feature_importances_ → important features

Main concept:

Pipelines make ML workflows safer, cleaner, and production-ready

In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_features     = ['Age', 'Fare', 'FamilySize', 'Pclass']
categorical_features = ['Sex', 'Embarked', 'Title', 'IsAlone', 'Has_Cabin']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

full_pipeline.fit(X_train, y_train)
y_pred2 = full_pipeline.predict(X_val)
print("Full pipeline accuracy:", round(accuracy_score(y_val, y_pred2), 4))

Full pipeline accuracy: 0.7933


ColumnTransformer:


applies different preprocessing to different column groups


Numeric pipeline:


median imputation


scaling


Categorical pipeline:


mode imputation


one-hot encoding


OneHotEncoder:


converts categories into binary columns


Main concept:
Different data types require different preprocessing workflows, and ColumnTransformer organizes them cleanly inside one pipeline

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth':    [4, 6, 8],
    'model__min_samples_leaf': [1, 2],
}

grid = GridSearchCV(full_pipeline, param_grid, cv=5,
                    scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", round(grid.best_score_, 4))

y_pred3 = grid.predict(X_val)
print("Tuned pipeline val accuracy:", round(accuracy_score(y_val, y_pred3), 4))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best params: {'model__max_depth': 4, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}
Best CV score: 0.8343
Tuned pipeline val accuracy: 0.7933


This is:

End-to-end ML workflow

Everything combined into:

preprocessing
encoding
scaling
model
tuning
evaluation

inside one reproducible pipeline.

This is how professional ML systems are usually built.

GridSearch can tune full pipelines directly.

Double underscore syntax:

step__parameter

Example:

model__max_depth

Benefits:


no leakage


cleaner workflow


safer CV


production-ready ML pipeline


Main concept:
Modern ML pipelines combine preprocessing, modeling, and tuning into one unified reproducible system

In [6]:
import joblib

joblib.dump(grid.best_estimator_, 'titanic_pipeline.pkl')
print("Pipeline saved.")

# Load it back and confirm it works
loaded_pipeline = joblib.load('titanic_pipeline.pkl')
y_check = loaded_pipeline.predict(X_val)
print("Loaded pipeline accuracy:", round(accuracy_score(y_val, y_check), 4))



Pipeline saved.
Loaded pipeline accuracy: 0.7933


joblib.dump():

saves trained pipeline/model to disk

joblib.load():

reloads saved pipeline

Saved pipeline contains:

preprocessing
tuning
trained model

Main concept:

ML models are usually trained once and then deployed/reused through saved serialized pipelines